
Dynamic Pricing and Inventory Optimization Engine
Week 1, Day 1: Data Cleaning and Preparation

Author: [Maithilee Dolharkar]
Date: [23 January 2026]

In [2]:
"""
Dynamic Pricing and Inventory Optimization Engine
Week 1, Day 1: Data Cleaning and Preparation

Author: [Your Name]
Date: [Today's Date]
Description: Load raw retail transaction data and perform data quality checks and cleaning
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os

warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

# Directory structure
directories = [
    'data/raw',
    'data/processed',
    'data/features',
    'results/visualizations',
    'models',
    'docs'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("Project directories created successfully")
print("Libraries imported successfully")


Project directories created successfully
Libraries imported successfully


In [3]:
"""
Load the raw UCI Online Retail II dataset
"""

# Load dataset
file_path = 'Downloads/retaildataset/online_retail_II.csv'

try:
    df_raw = pd.read_csv(file_path, encoding='ISO-8859-1')
    print(f"Dataset loaded successfully: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found. Please ensure the file is in the correct directory.")
    raise

# Display basic information
print("\n" + "="*70)
print("RAW DATASET OVERVIEW")
print("="*70)
print(f"\nShape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")

Dataset loaded successfully: 1,067,371 rows, 8 columns

RAW DATASET OVERVIEW

Shape: (1067371, 8)

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [4]:
"""
Inspect the first few rows and identify potential issues
"""

print("\n" + "="*70)
print("FIRST 10 ROWS")
print("="*70)
print(df_raw.head(10))

print("\n" + "="*70)
print("DATASET INFORMATION")
print("="*70)
df_raw.info()

print("\n" + "="*70)
print("STATISTICAL SUMMARY")
print("="*70)
print(df_raw.describe())


FIRST 10 ROWS
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
5  489434     22064           PINK DOUGHNUT TRINKET POT         24   
6  489434     21871                  SAVE THE PLANET MUG        24   
7  489434     21523   FANCY FONT HOME SWEET HOME DOORMAT        10   
8  489435     22350                            CAT BOWL         12   
9  489435     22349       DOG BOWL , CHASING BALL DESIGN        12   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95     13085.00  United Kingdom  
1  2009-12-01 07:45:00   6.75     13085.00  United Kingdom  
2  2009-12-01 07:45:00   6.75  

In [8]:
"""
Identify data quality issues: missing values, duplicates, invalid entries
"""

print("\n" + "="*70)
print("DATA QUALITY ASSESSMENT")
print("="*70)

# Missing values
print("\nMissing Values:")
missing_values = df_raw.isnull().sum()
missing_percentage = (missing_values / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing_Count': missing_values.values,
    'Percentage': missing_percentage.values
})
print(missing_df[missing_df['Missing_Count'] > 0])

# Duplicate rows
duplicates = df_raw.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates:,} ({duplicates/len(df_raw)*100:.2f}%)")

# Check for cancelled invoices (Invoice starting with 'C')
if 'Invoice' in df_raw.columns:
    cancelled_count = df_raw['Invoice'].astype(str).str.startswith('C').sum()
    print(f"\nCancelled Invoices: {cancelled_count:,} ({cancelled_count/len(df_raw)*100:.2f}%)")

# Check for negative quantities (returns)
if 'Quantity' in df_raw.columns:
    negative_qty = (df_raw['Quantity'] < 0).sum()
    print(f"Negative Quantities: {negative_qty:,} ({negative_qty/len(df_raw)*100:.2f}%)")

# Check for invalid prices
if 'Price' in df_raw.columns:
    zero_price = (df_raw['Price'] == 0).sum()
    negative_price = (df_raw['Price'] < 0).sum()
    print(f"Zero Prices: {zero_price:,} ({zero_price/len(df_raw)*100:.2f}%)")
    print(f"Negative Prices: {negative_price:,} ({negative_price/len(df_raw)*100:.2f}%)")


DATA QUALITY ASSESSMENT

Missing Values:
        Column  Missing_Count  Percentage
2  Description           4382        0.41
6  Customer ID         243007       22.77

Duplicate Rows: 34,335 (3.22%)

Cancelled Invoices: 19,494 (1.83%)
Negative Quantities: 22,950 (2.15%)
Zero Prices: 6,202 (0.58%)
Negative Prices: 5 (0.00%)


In [22]:
"""
Clean the dataset by removing invalid records
"""

print("\n" + "="*70)
print("DATA CLEANING PROCESS")
print("="*70)

# Create a copy for cleaning
df_clean = df_raw.copy()
initial_rows = len(df_clean)

print(f"\nStarting rows: {initial_rows:,}")

# Step 1: Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')
invalid_dates = df_clean['InvoiceDate'].isnull().sum()
if invalid_dates > 0:
    print(f"\nStep 1: Removing {invalid_dates:,} rows with invalid dates")
    df_clean = df_clean[df_clean['InvoiceDate'].notna()]
else:
    print(f"\nStep 1: All dates valid, no rows removed")

# Step 2: Remove cancelled invoices
rows_before = len(df_clean)
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
removed = rows_before - len(df_clean)
print(f"Step 2: Removed {removed:,} cancelled invoices")

# Step 3: Remove negative quantities (returns)
rows_before = len(df_clean)
df_clean = df_clean[df_clean['Quantity'] > 0]
removed = rows_before - len(df_clean)
print(f"Step 3: Removed {removed:,} rows with negative quantities")

# Step 4: Remove zero or negative prices
rows_before = len(df_clean)
df_clean = df_clean[df_clean['Price'] > 0]
removed = rows_before - len(df_clean)
print(f"Step 4: Removed {removed:,} rows with invalid prices")

# Step 5: Remove missing StockCode
rows_before = len(df_clean)
df_clean = df_clean[df_clean['StockCode'].notna()]
removed = rows_before - len(df_clean)
print(f"Step 5: Removed {removed:,} rows with missing StockCode")

# Step 6: Remove missing Description
rows_before = len(df_clean)
df_clean = df_clean[df_clean['Description'].notna()]
removed = rows_before - len(df_clean)
print(f"Step 6: Removed {removed:,} rows with missing Description")

# Step 7: Remove duplicate rows
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
removed = rows_before - len(df_clean)
if removed > 0:
    print(f"Step 7: Removed {removed:,} duplicate rows")

# Rename Customer ID column to remove space
df_clean = df_clean.rename(columns={'Customer ID': 'CustomerID'})

# Calculate TotalAmount
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['Price']

# Sort by date
df_clean = df_clean.sort_values('InvoiceDate').reset_index(drop=True)

final_rows = len(df_clean)
retention_rate = (final_rows / initial_rows) * 100

print("\n" + "="*70)
print("CLEANING SUMMARY")
print("="*70)
print(f"Initial rows: {initial_rows:,}")
print(f"Final rows: {final_rows:,}")
print(f"Rows removed: {initial_rows - final_rows:,}")
print(f"Retention rate: {retention_rate:.2f}%")


DATA CLEANING PROCESS

Starting rows: 1,067,371

Step 1: All dates valid, no rows removed
Step 2: Removed 19,494 cancelled invoices
Step 3: Removed 3,457 rows with negative quantities
Step 4: Removed 2,750 rows with invalid prices
Step 5: Removed 0 rows with missing StockCode
Step 6: Removed 0 rows with missing Description
Step 7: Removed 33,757 duplicate rows

CLEANING SUMMARY
Initial rows: 1,067,371
Final rows: 1,007,913
Rows removed: 59,458
Retention rate: 94.43%


In [23]:
"""
Validate cleaned dataset
"""

print("\n" + "="*70)
print("CLEANED DATASET SUMMARY")
print("="*70)

print(f"\nDate Range:")
print(f"  Start: {df_clean['InvoiceDate'].min()}")
print(f"  End: {df_clean['InvoiceDate'].max()}")
print(f"  Duration: {(df_clean['InvoiceDate'].max() - df_clean['InvoiceDate'].min()).days} days")

print(f"\nUnique Values:")
print(f"  Invoices: {df_clean['Invoice'].nunique():,}")
print(f"  Products: {df_clean['StockCode'].nunique():,}")
print(f"  Customers: {df_clean['CustomerID'].nunique():,}")
print(f"  Countries: {df_clean['Country'].nunique()}")

print(f"\nQuantity Statistics:")
print(f"  Mean: {df_clean['Quantity'].mean():.2f}")
print(f"  Median: {df_clean['Quantity'].median():.2f}")
print(f"  Std: {df_clean['Quantity'].std():.2f}")
print(f"  Min: {df_clean['Quantity'].min():.2f}")
print(f"  Max: {df_clean['Quantity'].max():.2f}")

print(f"\nPrice Statistics:")
print(f"  Mean: ${df_clean['Price'].mean():.2f}")
print(f"  Median: ${df_clean['Price'].median():.2f}")
print(f"  Std: ${df_clean['Price'].std():.2f}")
print(f"  Min: ${df_clean['Price'].min():.2f}")
print(f"  Max: ${df_clean['Price'].max():.2f}")

print(f"\nTotal Revenue: ${df_clean['TotalAmount'].sum():,.2f}")

remaining_missing = df_clean.isnull().sum()
if remaining_missing.sum() > 0:
    print(f"\nRemaining Missing Values:")
    print(remaining_missing[remaining_missing > 0])
else:
    print(f"\nNo missing values remaining in critical columns")

print(f"\nSample of cleaned data:")
print(df_clean.head(10))


CLEANED DATASET SUMMARY

Date Range:
  Start: 2009-12-01 07:45:00
  End: 2011-12-09 12:50:00
  Duration: 738 days

Unique Values:
  Invoices: 40,077
  Products: 4,917
  Customers: 5,878
  Countries: 43

Quantity Statistics:
  Mean: 11.12
  Median: 4.00
  Std: 128.47
  Min: 1.00
  Max: 80995.00

Price Statistics:
  Mean: $4.07
  Median: $2.10
  Std: $50.43
  Min: $0.00
  Max: $25111.09

Total Revenue: $20,476,260.45

Remaining Missing Values:
CustomerID    228488
dtype: int64

Sample of cleaned data:
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
5  489434     22064           PINK DOUGHNUT TRINKET POT         24   
6  4

In [24]:
"""
Save cleaned dataset for downstream analysis
"""

output_path = 'data/processed/online_retail_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"\nCleaned dataset saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

print("\nData cleaning complete!")


Cleaned dataset saved to: data/processed/online_retail_cleaned.csv
File size: 93.46 MB

Data cleaning complete!
